In [1]:
# : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : :
# 1. ИМПОРТ БИБЛИОТЕК
# : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : :

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [2]:
# : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : :
# ЗАГРУЗКА ДАННЫХ (ОБА ЛИСТА)
# : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : :

import pandas as pd

# Загружаем оба листа
df_2009 = pd.read_excel('../online_retail_II.xlsx', sheet_name='Year 2009-2010')
df_2010 = pd.read_excel('../online_retail_II.xlsx', sheet_name='Year 2010-2011')

print("="*50)
print("ЗАГРУЗКА ДАННЫХ")
print("="*50)
print(f"Лист 2009-2010: {df_2009.shape[0]:,} строк, {df_2009.shape[1]} колонок")
print(f"Лист 2010-2011: {df_2010.shape[0]:,} строк, {df_2010.shape[1]} колонок")

# Объединяем в один датафрейм
df = pd.concat([df_2009, df_2010], ignore_index=True)

print(f"\nОбъединённые данные: {df.shape[0]:,} строк, {df.shape[1]} колонок")
print(f"\nДиапазон дат:")
print(f"   Min: {df['InvoiceDate'].min()}")
print(f"   Max: {df['InvoiceDate'].max()}")

ЗАГРУЗКА ДАННЫХ
Лист 2009-2010: 525,461 строк, 8 колонок
Лист 2010-2011: 541,910 строк, 8 колонок

Объединённые данные: 1,067,371 строк, 8 колонок

Диапазон дат:
   Min: 2009-12-01 07:45:00
   Max: 2011-12-09 12:50:00


In [3]:
# : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : :
# 3. ПРОФИЛИРОВАНИЕ ДАННЫХ
# : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : :

print("\n" + "="*50)
print("ПРОФИЛИРОВАНИЕ ДАННЫХ")
print("="*50)

print("\nТИПЫ ДАННЫХ:")
print(df.dtypes)

print("\nСТАТИСТИКА ПО ЧИСЛОВЫМ КОЛОНКАМ:")
print(df[['Quantity', 'Price']].describe())

print("\nПРОВЕРКА ПРОПУСКОВ:")
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Пропуски': missing, '%': missing_pct})
print(missing_df[missing_df['Пропуски'] > 0])

print("\nТОП-5 СТРАН ПО КОЛИЧЕСТВУ ЗАКАЗОВ:")
print(df['Country'].value_counts().head(5))

print("\nАНОМАЛИИ:")
print(f"Отрицательные Quantity: {(df['Quantity'] < 0).sum():,}")
print(f"Отрицательные Price: {(df['Price'] < 0).sum():,}")
print(f"Дубликаты строк: {df.duplicated().sum():,}")


ПРОФИЛИРОВАНИЕ ДАННЫХ

ТИПЫ ДАННЫХ:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object

СТАТИСТИКА ПО ЧИСЛОВЫМ КОЛОНКАМ:
           Quantity         Price
count  1.067371e+06  1.067371e+06
mean   9.938898e+00  4.649388e+00
std    1.727058e+02  1.235531e+02
min   -8.099500e+04 -5.359436e+04
25%    1.000000e+00  1.250000e+00
50%    3.000000e+00  2.100000e+00
75%    1.000000e+01  4.150000e+00
max    8.099500e+04  3.897000e+04

ПРОВЕРКА ПРОПУСКОВ:
             Пропуски          %
Description      4382   0.410541
Customer ID    243007  22.766873

ТОП-5 СТРАН ПО КОЛИЧЕСТВУ ЗАКАЗОВ:
Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Name: count, dtype: int64

АНОМАЛИИ:
Отрицательные Quantity: 22,950
Отрицательны

In [4]:
# : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : :
# 4. ОЧИСТКА ДАННЫХ
# : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : :

print("\n" + "="*50)
print("ОЧИСТКА ДАННЫХ")
print("="*50)

# 4.1 Копируем исходные данные
df_clean = df.copy()

# 4.2 Удаляем возвраты (Quantity < 0)
df_clean = df_clean[df_clean['Quantity'] > 0]
print("Удалены записи с Quantity < 0 (возвраты)")

# 4.3 Удаляем аномальные цены (Price < 0)
df_clean = df_clean[df_clean['Price'] > 0]
print("Удалены записи с Price < 0 (аномалии)")

# 4.4 Удаляем строки без Customer ID
df_clean = df_clean.dropna(subset=['Customer ID'])
print("Удалены записи без Customer ID")

# 4.5 Удаляем дубликаты
df_clean = df_clean.drop_duplicates()
print("Удалены дубликаты строк")




ОЧИСТКА ДАННЫХ
Удалены записи с Quantity < 0 (возвраты)
Удалены записи с Price < 0 (аномалии)
Удалены записи без Customer ID
Удалены дубликаты строк


In [5]:
# : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : :
# 5. ПРОВЕРКА КАЧЕСТВА ОЧИСТКИ
# : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : :

print("\n" + "="*50)
print("ПРОВЕРКА КАЧЕСТВА ОЧИСТКИ")
print("="*50)

print(f"Исходные данные:   {len(df):,} строк")
print(f"После очистки:     {len(df_clean):,} строк")
print(f"Удалено:           {len(df) - len(df_clean):,} строк ({((len(df)-len(df_clean))/len(df)*100):.1f}%)")

print(f"\nДетали очистки:")
print(f"   • Клиентов осталось:   {df_clean['Customer ID'].nunique():,}")
print(f"   • Транзакций осталось: {df_clean['Invoice'].nunique():,}")
print(f"   • Товаров осталось:    {df_clean['StockCode'].nunique():,}")

# Проверка отсутствия аномалий
print(f"\nФинальная проверка:")
print(f"   Quantity < 0: {(df_clean['Quantity'] < 0).sum()}")
print(f"   Price < 0: {(df_clean['Price'] < 0).sum()}")
print(f"   Customer ID пропуски: {df_clean['Customer ID'].isnull().sum()}")
print(f"   Дубликаты: {df_clean.duplicated().sum()}")


ПРОВЕРКА КАЧЕСТВА ОЧИСТКИ
Исходные данные:   1,067,371 строк
После очистки:     779,425 строк
Удалено:           287,946 строк (27.0%)

Детали очистки:
   • Клиентов осталось:   5,878
   • Транзакций осталось: 36,969
   • Товаров осталось:    4,631

Финальная проверка:
   Quantity < 0: 0
   Price < 0: 0
   Customer ID пропуски: 0
   Дубликаты: 0


In [6]:
# : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : :
# ДОБАВЛЕНИЕ ВЫЧИСЛЯЕМЫХ ПОЛЕЙ 
# сохраняю преемственность с проектом ИЗ POWER BI
# : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : :

df_clean['Invoice Date'] = pd.to_datetime(df_clean['InvoiceDate']).dt.date
df_clean['Invoice Time'] = pd.to_datetime(df_clean['InvoiceDate']).dt.time

df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']


In [7]:
# Первая покупка для каждого клиента через группировку
first_purchase = df_clean.groupby('Customer ID')['InvoiceDate'].min().reset_index()
first_purchase.columns = ['Customer ID', 'First Purchase Date']
df_clean = df_clean.merge(
    first_purchase,
    on='Customer ID',
    how='left'
)

In [8]:
# Для поля 'First Purchaise Date' добавлю день, месяц, неделю, год первой покупки
df_clean['First Purchase Day'] = df_clean['First Purchase Date'].dt.day
df_clean['First Purchase Week'] = df_clean['First Purchase Date'].dt.strftime('%Y-W%V')
df_clean['First Purchase Month'] = df_clean['First Purchase Date'].dt.strftime('%Y-%m')
df_clean['First Purchase Year'] = df_clean['First Purchase Date'].dt.year

In [9]:
# поля после первой покупки будут использованы для когорт
df_clean['First Purchase Date'] = pd.to_datetime(df_clean['First Purchase Date'])
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])


# Разница (в днях) между первой покупкой и текущей покупкой
df_clean['Days Since First Purchase']   = (df_clean['InvoiceDate'] - df_clean['First Purchase Date']).dt.days
df_clean['Weeks Since First Purchase']  = (df_clean['InvoiceDate'] - df_clean['First Purchase Date']).dt.days 
df_clean['Months Since First Purchase'] = (df_clean['InvoiceDate'].dt.year - df_clean['First Purchase Date'].dt.year) * 12 + \
                                          (df_clean['InvoiceDate'].dt.month - df_clean['First Purchase Date'].dt.month) 
df_clean['Years Since First Purchase']  = (df_clean['InvoiceDate'].dt.year - df_clean['First Purchase Date'].dt.year) 


In [10]:

# : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : :
# 5. СОХРАНЕНИЕ ОЧИЩЕННЫХ ДАННЫХ
# : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : : :

df_clean.to_csv('../data/processed/cleaned_retail.csv', index = False)

In [11]:
df_clean.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country', 'Invoice Date', 'Invoice Time',
       'Revenue', 'First Purchase Date', 'First Purchase Day',
       'First Purchase Week', 'First Purchase Month', 'First Purchase Year',
       'Days Since First Purchase', 'Weeks Since First Purchase',
       'Months Since First Purchase', 'Years Since First Purchase'],
      dtype='object')

In [13]:
df_clean.head(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Invoice Date,Invoice Time,Revenue,First Purchase Date,First Purchase Day,First Purchase Week,First Purchase Month,First Purchase Year,Days Since First Purchase,Weeks Since First Purchase,Months Since First Purchase,Years Since First Purchase
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-12-01,07:45:00,83.4,2009-12-01 07:45:00,1,2009-W49,2009-12,2009,0,0,0,0
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01,07:45:00,81.0,2009-12-01 07:45:00,1,2009-W49,2009-12,2009,0,0,0,0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01,07:45:00,81.0,2009-12-01 07:45:00,1,2009-W49,2009-12,2009,0,0,0,0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-12-01,07:45:00,100.8,2009-12-01 07:45:00,1,2009-W49,2009-12,2009,0,0,0,0
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12-01,07:45:00,30.0,2009-12-01 07:45:00,1,2009-W49,2009-12,2009,0,0,0,0
